In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import openpyxl

In [13]:
data = pd.read_csv("dataset/customer_shopping_data.csv")
data["revenue"] = data["quantity"] * data["price"]
data["invoice_date"] = pd.to_datetime(data["invoice_date"],format='mixed')

In [14]:
data.head(3)

,invoice_no,customer_id,gender,age,category,quantity,price,payment_method,invoice_date,shopping_mall,revenue
0,I138884,C241288,Female,28,Clothing,5,1500.40,Credit Card,2022-05-08,Kanyon,7502.00
1,I317333,C111565,Male,21,Shoes,3,1800.51,Debit Card,2021-12-12,Forum Istanbul,5401.53
2,I127801,C266599,Male,20,Clothing,1,300.08,Cash,2021-09-11,Metrocity,300.08


# Customers

##### Customer gender

In [5]:
gender = data.groupby(["gender"],as_index=False).agg(
    count_of_gender = ("gender","count")
)

gender

,gender,count_of_gender
0,Female,59482
1,Male,39975


##### Customer age

In [7]:
data["age_group"] = data["age"].map(lambda x: "[0-14] -> Children" if x<15 
                                    else "[15-24] -> Youth" if x<26 
                                    else "[25-64] -> Adults" if x<65 
                                    else "[65-99] -> Older Adults")

age_group = data.groupby(["age_group"],as_index=False).agg(
    age_groups = ("age_group","count")
).sort_values(by="age_groups",ascending=False).reset_index(drop=True)

age_group

,age_group,age_groups
0,[25-64] -> Adults,74671
1,[15-24] -> Youth,15359
2,[65-99] -> Older Adults,9427


In [8]:
data.columns

Index(['invoice_no', 'customer_id', 'gender', 'age', 'category', 'quantity',
       'price', 'payment_method', 'invoice_date', 'shopping_mall', 'revenue',
       'age_group'],
      dtype='object')

##### Top 10 customers based on sales performance

In [19]:
snapshot_date = pd.Timestamp("2024-01-01")

rfm_table = data.groupby(["customer_id"],as_index=False).agg(
    Recency = ("invoice_date",lambda x: (snapshot_date - x.max()).days),
    Frequency = ("quantity","sum"),
    Monetary = ("revenue","sum")
).sort_values(by="Recency",ascending=False)

rfm_table

,customer_id,Recency,Frequency,Monetary
78178,C421044,1095,3,365.94
78781,C437854,1095,2,60.60
4026,C112914,1095,4,83.68
43264,C238585,1095,5,7502.00
58428,C286477,1095,5,7502.00
...,...,...,...,...
31078,C199460,30,5,7502.00
48499,C255092,30,2,162.64
93145,C826617,30,2,20.92
19851,C163422,30,4,9602.72


In [16]:
data["invoice_date"].max()

Timestamp('2023-12-02 00:00:00')